# Centauro-Lite — treino no Kaggle

Fine-tuning QLoRA do Qwen3-1.7B sobre o Psych-101, comparado ao Centaur.

**Antes de rodar, confira duas configuracoes no painel da direita:**

1. **Accelerator** -> `GPU T4 x2`. **Nao use a P100**: o unsloth exige CUDA
   capability 7.0 ou maior, a T4 e 7.5 e a P100 e 6.0. E um piso rigido, nao uma
   recomendacao de desempenho - abaixo dele o unsloth nao carrega.
2. **Internet** → `On`. E necessario para instalar o unsloth e baixar o dataset e o modelo.

Ordem das celulas: instalar → preparar dados → **medir o baseline** → treinar → medir de
novo → comparar com o Minitaur.

A celula do baseline nao e opcional. Sem o numero do modelo sem treino, nao ha como
afirmar que o fine-tuning fez qualquer coisa — so haveria um numero solto sem regua.

Repositorio: https://github.com/Mathwesm/tcc_projeto.git

## 1. Conferir a GPU

Se isto falhar ou nao mostrar uma placa, o Accelerator nao esta ligado.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

## 2. Instalar

O instalador do unsloth resolve sozinho a combinacao de torch e CUDA da maquina, por
isso ele roda primeiro e sozinho. Depois o projeto e instalado a partir do proprio
`pyproject.toml`, que o Poetry mantem — o Kaggle usa `pip`, mas a lista de dependencias
continua sendo a mesma de sempre.

Leva alguns minutos. E normal aparecer aviso de conflito de versao no fim.

In [ ]:
# Uma T4 x2 expoe duas placas, e o unsloth nao lida bem com as duas ao mesmo tempo.
# Um modelo de 1.7B em 4-bit sobra numa T4 de 16 GB, entao restringir a primeira
# custa nada e evita um erro de multi-GPU no meio do treino.
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

!pip install -q --upgrade --force-reinstall --no-cache-dir unsloth unsloth_zoo

In [ ]:
import os
from pathlib import Path

WORK = Path("/kaggle/working")
REPO_DIR = WORK / "tcc_projeto"

if not REPO_DIR.exists():
    !git clone --depth 1 https://github.com/Mathwesm/tcc_projeto.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull --ff-only

os.chdir(REPO_DIR)
os.environ["PYTHONUTF8"] = "1"
!pip install -q -e . --no-deps
!pip install -q pydantic pydantic-settings loguru typer pyyaml datasets pandas
print("cwd:", Path.cwd())

## 3. Preparar os dados

Baixa o Psych-101 (859 MB), filtra os tres experimentos do estudo de caso, e fatia as
transcricoes em janelas.

O split de treino e validacao **nao e sorteado aqui**: e lido do manifesto que veio no
repositorio. Isso importa mais do que parece — a seed sozinha nao fixa um split, ela
fixa uma permutacao da ordem que a biblioteca `datasets` produziu naquele momento, e
essa ordem pode mudar entre versoes. Lendo o manifesto, o Kaggle avalia comprovadamente
os mesmos participantes que ficaram de fora no notebook local.

In [ ]:
!python -m centauro_lite prepare --reuse-splits

## 4. Baseline: o Qwen3 **sem** treino nenhum

Este e o ponto de partida. Todo ganho do fine-tuning se mede contra este numero, nao
contra o 0,44 do paper — aquele foi medido nos 160 experimentos com outro tokenizador.

In [ ]:
!python -m centauro_lite evaluate --label "qwen3-1.7b-base"

## 5. Treinar

Salvando em `/kaggle/working`, que sobrevive enquanto a sessao estiver viva e vai junto
quando voce salva a versao do notebook. O disco temporario da maquina, nao.

Se der `CUDA out of memory`, a ordem de ajuste esta no README: primeiro reduzir
`max_seq_length` no `configs/default.yaml` (e rodar a celula 3 de novo, porque os dados
sao tokenizados naquele tamanho), depois trocar para `Qwen3-0.6B`.

**Atencao:** se mudar `max_seq_length`, o baseline da celula 4 tem que ser medido de
novo no mesmo tamanho. Janela maior melhora a nota por dar mais contexto, nao por
treinar melhor — comparar tamanhos diferentes invalida a conclusao.

In [ ]:
import time

start = time.perf_counter()
!python -m centauro_lite train --output /kaggle/working/adapter
print(f"\nTreino levou {(time.perf_counter() - start) / 60:.1f} minutos")

## 6. Medir o modelo treinado

Mesmo codigo, mesmo split, mesma metrica. So o modelo mudou.

In [ ]:
!python -m centauro_lite evaluate --adapter /kaggle/working/adapter --label "qwen3-1.7b-centauro-lite"

## 7. Minitaur-8B: a comparacao que sustenta o TCC

O Minitaur e a versao de 8 bilhoes de parametros do Centaur, publicada pelos proprios
autores com a mesma receita.

Repare que sao **duas** celulas, nao uma. O dataset guarda ids de token, e um id
pertence a um vocabulario so: o id 2610 e "You" no Qwen3 e " askear" no Llama. Avaliar
o Minitaur sobre os dados tokenizados para o Qwen3 faria ele ler ruido, produzir um
numero plausivel e nao levantar erro nenhum. Por isso o `prepare` roda de novo com o
`configs/minitaur.yaml`: mesmos participantes (o fingerprint do split nao muda),
tokenizacao propria.

Ressalva que fica no texto do TCC: NLL por token nunca e perfeitamente comparavel entre
tokenizadores diferentes, porque cada um corta o texto em pedacos diferentes. Dar a cada
modelo o seu proprio vocabulario e o minimo, nao a solucao.

In [ ]:
!python -m centauro_lite prepare --config configs/minitaur.yaml

In [ ]:
!python -m centauro_lite evaluate --config configs/minitaur.yaml --label "minitaur-8b"

## 8. Tabela final

Todos os resultados foram acumulados no mesmo arquivo. Aqui eles aparecem lado a lado.

In [ ]:
import json
from pathlib import Path

results = json.loads(Path("outputs/eval_results.json").read_text(encoding="utf-8"))
measured = {k: v for k, v in results.items() if not k.startswith("_")}

print(f"{'modelo':<34} {'NLL':>8}  {'tokens':>10}")
print("-" * 56)
for name, entry in sorted(measured.items(), key=lambda item: item[1]["nll"]):
    print(f"{name:<34} {entry['nll']:>8.4f}  {entry['n_scored_tokens']:>10,}")

print()
print("Referencias do paper (160 experimentos, outro tokenizador - baliza, nao comparacao direta):")
print(f"{'Centaur 70B':<34} {0.44:>8.2f}")
print(f"{'modelos cognitivos especializados':<34} {0.56:>8.2f}")
print(f"{'Llama 3.1 70B sem ajuste':<34} {0.58:>8.2f}")

print()
print("Por experimento:")
for name, entry in sorted(measured.items()):
    print(f"\n  {name}")
    for experiment, value in sorted(entry["per_experiment"].items()):
        tokens = entry["per_experiment_tokens"][experiment]
        print(f"    {experiment:<36} {value:.4f}  ({tokens:,} tokens)")

## 9. Guardar o resultado

O adapter treinado e o JSON de resultados estao em `/kaggle/working`. Para nao perder
quando a sessao morrer, use **Save Version** no canto superior direito — os arquivos
ficam anexados aquela versao do notebook e podem ser baixados depois.

O adapter e pequeno (alguns MB): sao so as camadas LoRA, nao o modelo inteiro.

In [ ]:
!ls -lh /kaggle/working/adapter
!cat outputs/eval_results.json